In [ ]:
import importlib.metadata
import os
import subprocess
import sys
import tempfile
from pathlib import Path

if "tensorflow" in sys.modules and os.environ.get("TF_USE_LEGACY_KERAS") != "1":
    raise RuntimeError(
        "Restart the Colab session, then run this notebook from its first cell. "
        "NIF requires the legacy Keras setting before TensorFlow is imported."
    )
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")

COLAB_REQUIREMENTS = [
    "tensorflow==2.21.0",
    "tf-keras==2.21.0",
    "keras==3.13.2",
    "tensorflow-probability==0.25.0",
    "tensorflow-model-optimization==0.8.1",
    "numpy==2.1.3",
    "scipy==1.16.3",
    "matplotlib==3.10.0",
    "tensorboard==2.21.0",
    "absl-py==1.4.0",
]
NIF_COMMIT = "ad9113df14590189b12068b6b3fa7c8c8d9187d4"

if sys.version_info < (3, 11) or sys.version_info >= (3, 14):
    raise RuntimeError(
        "Use a Colab Python 3.11–3.13 runtime for this pinned environment."
    )

missing = []
for requirement in COLAB_REQUIREMENTS:
    package, expected = requirement.split("==")
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != expected:
        missing.append(requirement)
if missing:
    package_modules = {
        "tensorflow": "tensorflow",
        "tf-keras": "tf_keras",
        "keras": "keras",
        "numpy": "numpy",
        "scipy": "scipy",
        "matplotlib": "matplotlib",
        "tensorflow-probability": "tensorflow_probability",
        "tensorflow-model-optimization": "tensorflow_model_optimization",
        "tensorboard": "tensorboard",
        "absl-py": "absl",
    }
    changing = {item.split("==")[0] for item in missing}
    loaded = [
        module
        for package, module in package_modules.items()
        if package in changing and module in sys.modules
    ]
    if loaded:
        raise RuntimeError(
            "Restart the Colab session before changing packages; already imported: "
            + ", ".join(loaded)
        )
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--disable-pip-version-check",
            *COLAB_REQUIREMENTS,
        ]
    )

runtime_root = Path(tempfile.gettempdir()) / ("nif-tutorials-v2-" + NIF_COMMIT[:12])
stamp = runtime_root / ".nif-source-commit"
if not stamp.exists() or stamp.read_text().strip() != NIF_COMMIT:
    runtime_root.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-deps",
            "--upgrade",
            "--disable-pip-version-check",
            "--target",
            str(runtime_root),
            "git+https://github.com/pswpswpsw/nif.git@" + NIF_COMMIT,
        ]
    )
    stamp.write_text(NIF_COMMIT)


def replace_compatibility(path, old, new):
    content = path.read_text(encoding="utf-8")
    if new in content:
        return
    if old in content:
        path.write_text(content.replace(old, new), encoding="utf-8")
    else:
        raise RuntimeError("Pinned NIF source changed unexpectedly: " + str(path))


optimizer_file = runtime_root / "nif" / "optimizers" / "external_optimizers.py"
replace_compatibility(
    optimizer_file,
    "from tensorflow.python.distribute import distribution_strategy_context as distribute_ctx",
    "from tensorflow.python.distribute import distribute_lib as distribute_ctx",
)
for optimizer_class in ("AdaBeliefOptimizer", "L4Adam"):
    replace_compatibility(
        optimizer_file,
        "class " + optimizer_class + "(tf.keras.optimizers.Optimizer):",
        "class " + optimizer_class + "(tf.keras.optimizers.legacy.Optimizer):",
    )
for name in ("lbfgs.py", "lbfgs_V2.py"):
    path = runtime_root / "nif" / "optimizers" / name
    content = path.read_text(encoding="utf-8")
    if "np.product(" in content:
        path.write_text(content.replace("np.product(", "np.prod("), encoding="utf-8")

tfr_file = runtime_root / "nif" / "data" / "tfr_dataset.py"
replace_compatibility(
    tfr_file,
    "target = tf.transpose(\n"
    "                batch_file[self.n_feature : self.n_feature + self.n_target]\n            )",
    "target = tf.transpose(\n"
    "                tf.squeeze(tf.stack("
    "batch_file[self.n_feature : self.n_feature + self.n_target], 0), 1)\n"
    "            )",
)
replace_compatibility(
    tfr_file,
    "target = tf.transpose(batch_file[-self.n_target :])",
    "target = tf.transpose(\n"
    "                tf.squeeze(tf.stack(batch_file[-self.n_target :], 0), 1)\n            )",
)

gc_file = runtime_root / "nif" / "optimizers" / "gtcf.py"
replace_compatibility(
    gc_file,
    "    def get_centralized_gradients_for_optimizer(loss, params):\n",
    """    if not getattr(optimizer, "_nif_gradient_centralization", False):
        original_get_gradients = optimizer._get_gradients

        def centralized_tape_gradients(*args, **kwargs):
            gradients_and_variables = original_get_gradients(*args, **kwargs)
            return [
                (gradient - tf.reduce_mean(
                    gradient, axis=list(range(len(gradient.shape) - 1)), keepdims=True
                )
                if gradient is not None and len(gradient.shape) > 1 else gradient, variable)
                for gradient, variable in gradients_and_variables
            ]

        optimizer._get_gradients = centralized_tape_gradients
        optimizer._nif_gradient_centralization = True

    def get_centralized_gradients_for_optimizer(loss, params):
""",
)

if str(runtime_root) not in sys.path:
    sys.path.insert(0, str(runtime_root))
if "nif" in sys.modules:
    loaded_nif = Path(sys.modules["nif"].__file__).resolve()
    if not loaded_nif.is_relative_to(runtime_root.resolve()):
        raise RuntimeError(
            "Restart the session: an unrelated NIF installation is already imported."
        )
print("Python:", sys.version.split()[0])
for requirement in COLAB_REQUIREMENTS:
    package = requirement.split("==")[0]
    print(package + ":", importlib.metadata.version(package))
print("NIF source commit:", NIF_COMMIT)
print("Compatibility copy:", runtime_root / "nif")


In [ ]:
import tensorflow as tf
import tensorflow_model_optimization as tfmot
import nif
import numpy as np
import time
import logging
import contextlib
from matplotlib import pyplot as plt

# Model pruning

In this tutorial, you will learn how to prune/quantanzation your NIF model using [tensorflow_model_optimization](https://www.tensorflow.org/model_optimization). 

## First, let's train a model to start with

In [ ]:
cfg_shape_net = {
    "use_resblock":False,
    "connectivity": 'full',
    "input_dim": 1,
    "output_dim": 1,
    "units": 30,
    "nlayers": 2,
    "weight_init_factor": 0.01,
    "omega_0":30.0
}
cfg_parameter_net = {
    "use_resblock":False,
    "input_dim": 1,
    "latent_dim": 1,
    "units": 30,
    "nlayers": 2,
    "activation": 'swish',
}

class LossAndErrorPrintingCallback(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.train_begin_time = time.time()
        self.history_loss = []
        logging.basicConfig(filename='./log_multiscale', level=logging.INFO, format='%(message)s')

    def on_epoch_begin(self, epoch, logs=None):
        self.ts = time.time()

    def on_epoch_end(self, epoch, logs=None):
        
        if epoch % display_epoch == 0:
            tnow = time.time()
            te = tnow - self.ts
            logging.info("Epoch {:6d}: avg.loss pe = {:4.3e}, {:d} points/sec, time elapsed = {:4.3f} hours".format(
                epoch, logs['loss'], int(batch_size / te), (tnow - self.train_begin_time) / 3600.0))
            self.history_loss.append(logs['loss'])
        
        if epoch % print_figure_epoch == 0:
            plt.figure()
            plt.semilogy(self.history_loss)
            plt.xlabel('epoch: per {} epochs'.format(print_figure_epoch))
            plt.ylabel('MSE loss')
            plt.savefig('./loss_multiscale.png')
            plt.close()

        if epoch % checkpt_epoch == 0 or epoch == nepoch - 1:
            print('save checkpoint epoch: %d...' % epoch)
            self.model.save_weights("./saved_weights_multiscale/ckpt-{}/ckpt".format(epoch))

lr = 1e-3
def scheduler(epoch, lr):
    if epoch < 1000:
        return lr
    elif epoch < 2000:
        return 1e-4
    elif epoch < 4000:
        return 5e-5
    else:
        return 1e-5   

scheduler_callback = tf.keras.callbacks.LearningRateScheduler(scheduler)

enable_multi_gpu = False
enable_mixed_precision = False
nepoch = 5000
lr = 1e-3
batch_size = 512
checkpt_epoch = 1000
display_epoch = 100
print_figure_epoch = 100

Let's open up the tensorboard to monitor the result

In [ ]:
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir="./tb-logs-no-reg", update_freq='epoch')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir .

In [ ]:
# get training data set
from nif.demo import TravelingWaveHighFreq
tw = TravelingWaveHighFreq()

# take 
raw_data = tw.data
np.random.shuffle(raw_data)

train_data = raw_data
# train_data = raw_data[:int(raw_data.shape[0]*0.9)]
# valid_data = raw_data[int(raw_data.shape[0]*0.9):]

num_total_data = train_data.shape[0]
train_dataset = tf.data.Dataset.from_tensor_slices((train_data[:, :2],
                                                    train_data[:, 2:3]))
train_dataset = train_dataset.shuffle(num_total_data).batch(batch_size).prefetch(tf.data.experimental.AUTOTUNE)

# valid_dataset = tf.data.Dataset.from_tensor_slices((valid_data[:, :2],
#                                                      valid_data[:, 2:3]))

# mixed precision?
if enable_mixed_precision:
    mixed_policy = "mixed_float16"
    # we might need this for `model.fit` to automatically do loss scaling
    policy = nif.mixed_precision.Policy(mixed_policy)
    nif.mixed_precision.set_global_policy(policy)
else:
    mixed_policy = 'float32'

# cm = tf.distribute.MirroredStrategy().scope() if enable_multi_gpu else contextlib.nullcontext()
# with cm:
optimizer = tf.keras.optimizers.Adam(lr)
model_ori = nif.NIFMultiScale(cfg_shape_net, cfg_parameter_net, mixed_policy)
model = model_ori.build()
model.compile(optimizer, loss='mse')

model.summary()

callbacks = [tensorboard_callback, LossAndErrorPrintingCallback(), scheduler_callback]
model.fit(train_dataset, epochs=nepoch, verbose=0, callbacks=callbacks)

### we save the prediction of trained model

In [ ]:
t = np.linspace(-1,1,10,endpoint=True)
x = np.linspace(-1,1,200,endpoint=True)

tt,xx=np.meshgrid(t,x)

input_feature = np.hstack([tt.reshape(-1,1),xx.reshape(-1,1)])

model = model_ori.build()
model.load_weights('./saved_weights_multiscale/ckpt-4999/ckpt')

y_pred_unpruned = model.predict(input_feature).reshape(200,10)

# plt.contourf(tt,xx,y_pred_unpruned,levels=np.linspace(-1,1,50),cmap='seismic')

### We save this model as HDF5 format

In [ ]:
keras_file = './original.h5'
tf.keras.models.save_model(model, keras_file, include_optimizer=False)

## Second, let's load this model from disk and start pruning

- here we prune the ParameterNet only.
- only weights are pruned.
- you can choose to prune either the last layer of ParameterNet OR the whole ParameterNet
- when pruning, we use mag. based pruning criterion - low mag. weights are pruned

About pruning setup
- we use `tfmot.sparsity.keras.PolynomialDecay` to prune 10% weights, and gradually increasing to 90%.
- we first prune 10% then, train for 1000 epochs. We repeat 50 times of this process, each time we prune a bit more, until we hit the limit of 90% of included weights pruned.

In [ ]:
model_2 = model_ori.build()
model_2.load_weights('./saved_weights_multiscale/ckpt-4999/ckpt')

nepoch=5000*10
prune_epoch=100*10
end_step = np.ceil(num_total_data / batch_size).astype(np.int32) * nepoch
freq_step = np.ceil(num_total_data / batch_size).astype(np.int32) * prune_epoch
pruning_params = {'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(initial_sparsity=0.10,
                                                               final_sparsity=0.90,
                                                               begin_step=0,
                                                               end_step=end_step,
                                                               frequency=freq_step)
                 }

### Let's take a look each layer's name

In [ ]:
model_2.summary()

- Here we can choose to prune what we want... how about start with last layer of ParameterNet? 
- We can write a function to pull out this layer by just checking which layer has **SIREN** in its name

In [ ]:
def apply_pruning_to_dense(layer):
    if 'SIREN' in layer.name:
        print('layer selected = ', layer.name)
        return tfmot.sparsity.keras.prune_low_magnitude(layer,**pruning_params)
    return layer

# or we can prune every layer of ParameterNet.. it's up to you
# def apply_pruning_to_dense(layer):
#     if 'SIREN' in layer.name or 'pnet' in layer.name:
#         print('layer selected = ', layer.name)
#         return tfmot.sparsity.keras.prune_low_magnitude(layer,**pruning_params)
#     return layer


- get a redefined model for pruning

In [ ]:
model_for_pruning = tf.keras.models.clone_model(model_2, clone_function=apply_pruning_to_dense)

- `prune_low_magnitude` requires a recompile.

In [ ]:
optimizer = tf.keras.optimizers.Adam(1e-4)
model_for_pruning.compile(optimizer=optimizer,loss='mse')

- Let's check the model, you will see the layer being pruned will have a prefix `pruned`

In [ ]:
model_for_pruning.summary()

- Ok, now let's just prune it - first prune, then retrain, then prune....

In [ ]:
## training with pruning.

log_dir = 'tb-logs-prune-0.90'
callbacks = [tfmot.sparsity.keras.UpdatePruningStep(),
             tfmot.sparsity.keras.PruningSummaries(log_dir=log_dir)]

history = model_for_pruning.fit(train_dataset, epochs=nepoch, callbacks=callbacks)

### Finally, we finished pruning. Let's see how mini-batch loss changing over epoch! 

In [ ]:
plt.figure(figsize=(16,8))
plt.semilogy(history.history['loss'],'k-')
plt.rcParams['figure.facecolor'] = 'white'

for i in range(50):
    plt.plot([i*prune_epoch,i*prune_epoch],[1e-5,5e-1],'r-',alpha=0.25)

plt.text(-1000, 7e-1, r'10\%',size=25,c='r')
plt.text(48000, 7e-1, r'90\%',size=25,c='r')

plt.xlabel('epoch',size=25)
plt.ylabel('train err',size=25)
plt.xticks(fontsize=20);
plt.yticks(fontsize=20);
plt.xlim([-1500,52000])
plt.ylim([1e-6,3])
plt.savefig('pruned-learning-curve-10-90.png',bbox_inches='tight')

### analyze the weights

- let's verify the size of the pruned layer, see if 90% of the weights are zero?

In [ ]:
hyper_siren_layer = model_for_pruning.get_layer('prune_low_magnitude_HyperLinearForSIREN')
print("hyper_siren_layer has {} weights, only {} of them are not zero. ".format(hyper_siren_layer.weights[0].shape[1],hyper_siren_layer.weights[2].numpy().sum()))

- we make prediction on the pruned model

In [ ]:
t = np.linspace(-1,1,10,endpoint=True)
x = np.linspace(-1,1,200,endpoint=True)

tt, xx = np.meshgrid(t,x)
input_feature = np.hstack([tt.reshape(-1,1),xx.reshape(-1,1)])

y_pred = model_for_pruning.predict(input_feature).reshape(200,10)

### after pruning, let's save stripped model to file

In [ ]:
model_for_export = tfmot.sparsity.keras.strip_pruning(model_for_pruning)
pruned_keras_file = './stripped.h5'
tf.keras.models.save_model(model_for_export, pruned_keras_file, include_optimizer=False)

### for further compression, we convert it to tflite

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model_for_export)
pruned_tflite_model = converter.convert()
pruned_tflite_file = './compressed.tflite'
with open(pruned_tflite_file, 'wb') as f:
    f.write(pruned_tflite_model)

### Now, we can take a look at how much compression we got

In [ ]:
def get_gzipped_model_size(file):
    # Returns size of gzipped model, in bytes.
    import os
    import zipfile
    import tempfile

    _, zipped_file = tempfile.mkstemp('.zip')
    with zipfile.ZipFile(zipped_file, 'w', compression=zipfile.ZIP_DEFLATED) as f:
        f.write(file)

    return os.path.getsize(zipped_file)

## Finally, let us try quantazation to maximally compress the size

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model_for_export)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
quantized_and_pruned_tflite_model = converter.convert()

quantized_and_pruned_tflite_file = 'quantized_and_pruned.tflite'

with open(quantized_and_pruned_tflite_file, 'wb') as f:
    f.write(quantized_and_pruned_tflite_model)

print("Size of gzipped baseline Keras model: %.2f bytes" % (get_gzipped_model_size(keras_file)))
print("====")
print("Size of gzipped pruned Keras model: %.2f bytes" % (get_gzipped_model_size(pruned_keras_file)))
print("Size of gzipped pruned TFlite model: %.2f bytes" % (get_gzipped_model_size(pruned_tflite_file)))
print("Size of gzipped pruned and quantized TFlite model: %.2f bytes" % (get_gzipped_model_size(quantized_and_pruned_tflite_file)))

In [ ]:
import numpy as np

def evaluate_model(interpreter):
    input_index = interpreter.get_input_details()[0]["index"]
    output_index = interpreter.get_output_details()[0]["index"]

    # Run predictions on ever y image in the "test" dataset.
    prediction = []
    for i, test_xt in enumerate(input_feature):
        # Pre-processing: add batch dimension and convert to float32 to match with
        # the model's input data format.
        test_xt_ = np.expand_dims(test_xt, axis=0).astype(np.float32)
        # print(test_xt_)
        interpreter.set_tensor(input_index, test_xt_)

        # Run inference.
        interpreter.invoke()

        # Post-processing: remove batch dimension and find the digit with highest
        # probability.
        output = interpreter.get_tensor(output_index)
        prediction.append(output)

    print('\n')
    # Compare prediction results with ground truth labels to calculate accuracy.
    prediction = np.vstack(prediction).reshape(200,10)
    return prediction

In [ ]:
interpreter = tf.lite.Interpreter(model_content=quantized_and_pruned_tflite_model)
interpreter.allocate_tensors()

y_pred_quant = evaluate_model(interpreter)

In [ ]:
t = np.linspace(-1,1,10,endpoint=True)
x = np.linspace(-1,1,200,endpoint=True)

tt,xx=np.meshgrid(t,x)

input_feature = np.hstack([tt.reshape(-1,1),xx.reshape(-1,1)])

# plt.contourf(tt,xx,y_pred_quant,levels=np.linspace(-1,1,50),cmap='seismic')
# plt.savefig('compressed_pred_quant.png',bbox_inches='tight')

In [ ]:
np.linalg.norm(y_pred_quant-y_pred_unpruned,ord=2)**2/np.linalg.norm(y_pred_unpruned,ord=2)**2

# Finally, let's compare the compressed result! 

In [ ]:
# plt.figure()
fig, axs = plt.subplots(3,2,figsize=(16,16))

# pruned - 90%
axs[0,0].set_title('reconstruction - unpruned ')
axs[0,0].contourf(tt,xx,y_pred_unpruned,levels=np.linspace(-1,1,50),cmap='seismic')
axs[1,0].set_title('reconstruction - pruned 90%')
axs[1,0].contourf(tt,xx,y_pred,levels=np.linspace(-1,1,50),cmap='seismic')
axs[2,0].set_title('difference')
axs[2,0].contourf(tt,xx,y_pred-y_pred_unpruned,levels=np.linspace(-1,1,50),cmap='seismic')

# pruned - 90% + quant
axs[0,1].set_title('reconstruction - unpruned ')
axs[0,1].contourf(tt,xx,y_pred_unpruned,levels=np.linspace(-1,1,50),cmap='seismic')
axs[1,1].set_title('reconstruction - pruned 90% + quant')
axs[1,1].contourf(tt,xx,y_pred_quant,levels=np.linspace(-1,1,50),cmap='seismic')
axs[2,1].set_title('difference')
axs[2,1].contourf(tt,xx,y_pred-y_pred_quant,levels=np.linspace(-1,1,50),cmap='seismic')



In [ ]:
print("relative MSE - prune is {}".format(np.linalg.norm(y_pred-y_pred_unpruned,ord=2)**2/np.linalg.norm(y_pred_unpruned,ord=2)**2))
print("relative MSE - prune+quant is {}".format(np.linalg.norm(y_pred_quant-y_pred_unpruned,ord=2)**2/np.linalg.norm(y_pred_unpruned,ord=2)**2))